In [10]:

import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')
path = '/kaggle/input/datasets/olistbr/brazilian-ecommerce/'

tables = {
    'orders':          'olist_orders_dataset.csv',
    'order_items':     'olist_order_items_dataset.csv',
    'products':        'olist_products_dataset.csv',
    'customers':       'olist_customers_dataset.csv',
    'cat_translation': 'product_category_name_translation.csv',
}

for name, file in tables.items():
    pd.read_csv(path + file).to_sql(name, conn, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql_query(sql, conn)

print("Loaded:", list(tables.keys()))

Loaded: ['orders', 'order_items', 'products', 'customers', 'cat_translation']


In [11]:
q("SELECT * FROM orders LIMIT 3")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [12]:
q("SELECT * FROM order_items LIMIT 3")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


## 1. Monthly Revenue Trend
Revenue grew almost 8x in 18 months, from about BRL 120k (Jan 2017) to about BRL 920k per month across 2018. November 2017 was the single biggest month (about BRL 1M), matching Black Friday and Brazil's "13th salary" season.

In [13]:
q("""  SELECT strftime('%Y-%m',order_purchase_timestamp) AS month,
SUM(price) AS revenue 
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY strftime('%Y-%m',order_purchase_timestamp)
ORDER BY month



""")

,month,revenue
0,2016-09,267.36
1,2016-10,49507.66
2,2016-12,10.90
3,2017-01,120312.87
4,2017-02,247303.02
5,2017-03,374344.30
6,2017-04,359927.23
7,2017-05,506071.14
8,2017-06,433038.60
9,2017-07,498031.48


In [14]:
q("SELECT * FROM cat_translation LIMIT 5")

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## 2. Top Product Categories by Revenue
Revenue is highly concentrated: of 71 categories, the top 5 each earn about BRL 0.9 to 1.3M, while a long tail of about 66 categories contribute very little. A handful of categories carry the business.

In [15]:
q(""" SELECT product_category_name_english AS category, SUM(price) AS revenue
FROM order_items oi
JOIN products p
ON oi.product_id = p.product_id
JOIN cat_translation ct
ON p.product_category_name = ct.product_category_name
GROUP BY product_category_name_english
ORDER BY revenue DESC






""")

,category,revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
...,...,...
66,flowers,1110.04
67,home_comfort_2,760.27
68,cds_dvds_musicals,730.00
69,fashion_childrens_clothes,569.85


## 3. Delivery Performance
Orders take about 12.6 days on average to arrive, and only about 8% arrive after the estimated date, so about 92% are on time or early. Strong reliability, though it may partly reflect conservative delivery estimates.

In [17]:
q(""" SELECT AVG(julianday(order_delivered_customer_date)- julianday(order_purchase_timestamp) )
AS avg_delivery_days
FROM orders 


""")

,avg_delivery_days
0,12.558702


In [19]:
q("""SELECT SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END)
* 100.0 / COUNT(*) AS pct_late
FROM orders
WHERE order_delivered_customer_date IS NOT NULL """ )

,pct_late
0,8.112899
